# Step 1: Parse PDF — Extract Text (with page numbers) & Images

In [ ]:
%pip install PyMuPDF

In [1]:
%pip install PyMuPDF -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import json

PDF_PATH = "harrypotter.pdf"
OUTPUT_DIR = "extracted_data"
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
MIN_IMAGE_SIZE = 5000  # Skip tiny decorative images (< 5KB)

os.makedirs(IMAGES_DIR, exist_ok=True)
doc = fitz.open(PDF_PATH)
print(f"Opened: {PDF_PATH} — {len(doc)} pages\n")

# --- Extract text + images in one pass ---
pages = []
total_images = 0

for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    text = page.get_text().strip()
    
    # Check for images on this page
    page_images = []
    for img_index, img in enumerate(page.get_images()):
        xref = img[0]
        base_image = doc.extract_image(xref)
        if len(base_image["image"]) < MIN_IMAGE_SIZE:
            continue  # skip tiny icons/bullets
        
        filename = f"page_{page_num + 1}_img_{img_index + 1}.{base_image['ext']}"
        filepath = os.path.join(IMAGES_DIR, filename)
        with open(filepath, "wb") as f:
            f.write(base_image["image"])
        
        page_images.append({
            "filename": filename,
            "width": base_image["width"],
            "height": base_image["height"],
        })
    
    total_images += len(page_images)
    pages.append({
        "page_number": page_num + 1,
        "text": text,
        "has_images": len(page_images) > 0,
        "images": page_images,
    })

doc.close()

# --- Summary ---
pages_with_images = [p for p in pages if p["has_images"]]
print(f"Text extracted: {len(pages)} pages")
print(f"Images found: {total_images} images across {len(pages_with_images)} pages")
print(f"Pages with images: {[p['page_number'] for p in pages_with_images]}")

Opened: harrypotter.pdf — 3623 pages

Text extracted: 3623 pages
Images found: 218 images across 217 pages
Pages with images: [1, 3, 5, 8, 12, 26, 37, 50, 55, 62, 85, 107, 123, 133, 150, 151, 164, 176, 194, 205, 217, 222, 234, 257, 277, 282, 291, 301, 316, 335, 353, 368, 383, 399, 417, 434, 453, 471, 489, 502, 517, 536, 553, 568, 573, 584, 585, 598, 613, 630, 653, 675, 690, 708, 726, 749, 768, 785, 799, 818, 837, 852, 867, 874, 891, 898, 924, 942, 949, 961, 969, 979, 989, 1001, 1009, 1026, 1044, 1067, 1078, 1089, 1107, 1121, 1137, 1154, 1174, 1188, 1196, 1210, 1230, 1252, 1270, 1285, 1310, 1330, 1348, 1372, 1394, 1419, 1434, 1454, 1479, 1486, 1498, 1507, 1525, 1545, 1563, 1570, 1588, 1609, 1626, 1645, 1663, 1685, 1700, 1714, 1741, 1761, 1781, 1810, 1838, 1864, 1887, 1906, 1908, 1929, 1952, 1974, 1994, 2018, 2043, 2066, 2092, 2099, 2118, 2129, 2145, 2169, 2195, 2219, 2245, 2269, 2290, 2303, 2320, 2345, 2358, 2382, 2409, 2415, 2431, 2447, 2462, 2482, 2503, 2523, 2545, 2559, 2578, 2597, 2

In [3]:
# Save to JSON for the next notebook (chunking + embedding)
output_path = os.path.join(OUTPUT_DIR, "parsed_pages.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(output_path) / 1024 / 1024
print(f"Saved: {output_path} ({file_size:.1f} MB)")
print(f"Images saved to: {IMAGES_DIR}/")
print(f"\nDone! Ready for Step 2.")

Saved: extracted_data\parsed_pages.json (6.7 MB)
Images saved to: extracted_data\images/

Done! Ready for Step 2.
